
The goal of this project is to predict whether a customer will churn or not using historical telecom customer data. This helps businesses take proactive retention actions and reduce customer loss.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
files.upload()

In [2]:
df=pd.read_csv('Processed_Churn_Data.csv')
df.head()

,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,churn_value,tenure_period
0,1,0,0,0,2,1,No,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,1,short
1,0,0,0,1,2,1,No,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,1,short
2,0,0,0,1,8,1,Yes,Fiber optic,0,0,1,0,1,1,Month-to-month,1,Electronic check,99.65,1,short
3,0,0,1,1,28,1,Yes,Fiber optic,0,0,1,1,1,1,Month-to-month,1,Electronic check,104.80,1,medium
4,1,0,0,1,49,1,Yes,Fiber optic,0,1,1,0,1,1,Month-to-month,1,Bank transfer (automatic),103.70,1,long


In [3]:
df.columns

Index(['gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months',
       'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'churn_value', 'tenure_period'],
      dtype='object')

### Checking the correleation once more

In [4]:
df.corr(numeric_only=True)['churn_value'].sort_values(ascending=False)

,churn_value
churn_value,1.000000
monthly_charges,0.193356
paperless_billing,0.191825
senior_citizen,0.150889
streaming_tv,0.063228
streaming_movies,0.061382
phone_service,0.011942
gender,-0.008612
device_protection,-0.066160
online_backup,-0.082255


####Dropping the newly made features for tenure_months to keep the original scale true for better understanding

In [5]:
df = df.drop(columns=['tenure_period_low','tenure_period_medium','tenure_period_long'], errors='ignore')

###Encoding
We haven't done encoding for categorical columns because we want to do it inside pipeline so that when we use new data and customer will provide original value then it will encode automatically.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure_months", "monthly_charges"]

cat_cols = [
    "gender", "partner", "dependents", "phone_service",
    "online_security", "online_backup", "device_protection",
    "tech_support", "streaming_tv", "streaming_movies",
    "paperless_billing", "contract", "payment_method",
    "internet_service", "multiple_lines"
]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

##Train Test Split

In [7]:
from sklearn.model_selection import train_test_split
X=df.drop('churn_value',axis=1)
y=df['churn_value']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

##Pipeline Building
In this we have taken multiple models with correct pipeline flow using scaling then selecting best feature and then appending them with models.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

pipeline={'Logistic Regression':Pipeline([("preprocessing", preprocessor),('feature_selection',SelectKBest(score_func=f_classif)),('model',LogisticRegression(class_weight='balanced'))]),
          'KNN':Pipeline([("preprocessing", preprocessor),('feature_selection',SelectKBest(score_func=f_classif)),('model',KNeighborsClassifier())]),
          'SVM':Pipeline([("preprocessing", preprocessor),('feature_selection',SelectKBest(score_func=f_classif)),('model',SVC(kernel='rbf'))]),
          'Random Forest':Pipeline([("preprocessing", preprocessor),('feature_selection',SelectKBest(score_func=f_classif)),('model',RandomForestClassifier(class_weight='balanced',random_state=42))])}

## Model Building/Cross Validation

Multiple classification algorithms were tested including:
- Logistic Regression  
- K-Nearest Neighbors  
- Support Vector Machine  
- Random Forest  

Models were evaluated using **Stratified K-Fold Cross Validation** with **recall as the primary metric** due to class imbalance.\
Recall was chosen as the primary evaluation metric because in churn prediction, it is more important to correctly identify customers who are likely to churn rather than minimizing false positives.

In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cv_score={}

for name,model in pipeline.items():
  scores=cross_val_score(model,X_train,y_train,cv=cv,scoring='recall')
  cv_score[name]=scores.mean()
  print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")

Logistic Regression: Mean=0.8087, Std=0.0308
KNN: Mean=0.5411, Std=0.0188
SVM: Mean=0.5425, Std=0.0198
Random Forest: Mean=0.5037, Std=0.0159


###Best pipeline
We have chosen **Logistic regression** as best model because it has **highest recall** which is most important thing for classification because in this **False negative is expensive** hence we can't flag a churning customer as non churn.

In [10]:
best_pipeline=pipeline['Logistic Regression']

##Hyperparameter Tuning
We are using Randomized Search CV for finding the best **hyperparameters** for our best model because it is fast and select randomly but provide best results.

We are not moving to more deep hyperparameter tuning using Grid Search CV because using RandomSearchCV has only improved recall to a little extent that means our model is at its peak.

Insted of improving model a very little using Grid search we are going to use ***Threshold Tuning**.

In [11]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'feature_selection__k': [5, 10, 15, 'all'],
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['lbfgs', 'liblinear'],
    'model__penalty': ['l2'],
}
random_search = RandomizedSearchCV(best_pipeline, param_distributions=param_dist, n_iter=10, cv=cv, scoring='recall', random_state=42,n_jobs=-1)
random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('num',
                                                                               StandardScaler(),
                                                                               ['tenure_months',
                                                                                'monthly_charges']),
                                                                              ('cat',
                                                                               OneHotEncoder(drop='first',
                                                                                             handle_unknown='ignore'),
                                                                               ['gender',
                                                                                'partner',
                                                                                'dependents',
                                                                                'phone_service',
                                                                                'online_security',
                                                                                'online_b...
                                                                                'contract',
                                                                                'payment_method',
                                                                                'internet_service',
                                                                                'multiple_lines'])])),
                                             ('feature_selection',
                                              SelectKBest()),
                                             ('model',
                                              LogisticRegression(class_weight='balanced'))]),
                   n_jobs=-1,
                   param_distributions={'feature_selection__k': [5, 10, 15,
                                                                 'all'],
                                        'model__C': [0.01, 0.1, 1, 10, 100],
                                        'model__penalty': ['l2'],
                                        'model__solver': ['lbfgs',
                                                          'liblinear']},
                   random_state=42, scoring='recall')

In [12]:
print("Best Params:", random_search.best_params_)
print("Best CV Recall:", random_search.best_score_)

Best Params: {'model__solver': 'lbfgs', 'model__penalty': 'l2', 'model__C': 10, 'feature_selection__k': 5}
Best CV Recall: 0.8187290969899665


In [13]:
best_model=random_search.best_estimator_
best_model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['tenure_months',
                                                   'monthly_charges']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['gender', 'partner',
                                                   'dependents',
                                                   'phone_service',
                                                   'online_security',
                                                   'online_backup',
                                                   'device_protection',
                                                   'tech_support',
                                                   'streaming_tv',
                                                   'streaming_movies',
                                                   'paperless_billing',
                                                   'contract', 'payment_method',
                                                   'internet_service',
                                                   'multiple_lines'])])),
                ('feature_selection', SelectKBest(k=5)),
                ('model', LogisticRegression(C=10, class_weight='balanced'))])

##Threshold tuning
Instead of using the default classification threshold of 0.5, different thresholds were tested to optimize recall.

Lower thresholds improved the model's ability to capture churn customers, which is critical for business decision-making.

We choose 0.3 because it provides best and balanced outputs.

In [14]:
from sklearn.metrics import classification_report
y_prob = best_model.predict_proba(X_test)[:, 1]
for t in [0.2, 0.3, 0.4, 0.5]:
    y_pred = (y_prob > t).astype(int)
    print(f"Threshold: {t}")
    print(classification_report(y_test, y_pred))

Threshold: 0.2
              precision    recall  f1-score   support

           0       0.97      0.44      0.60      1035
           1       0.38      0.96      0.54       374

    accuracy                           0.58      1409
   macro avg       0.67      0.70      0.57      1409
weighted avg       0.81      0.58      0.59      1409

Threshold: 0.3
              precision    recall  f1-score   support

           0       0.95      0.54      0.69      1035
           1       0.42      0.92      0.57       374

    accuracy                           0.64      1409
   macro avg       0.68      0.73      0.63      1409
weighted avg       0.81      0.64      0.66      1409

Threshold: 0.4
              precision    recall  f1-score   support

           0       0.93      0.62      0.74      1035
           1       0.45      0.86      0.59       374

    accuracy                           0.68      1409
   macro avg       0.69      0.74      0.67      1409
weighted avg       0.80      

## Final Model Performance

The final model was selected after hyperparameter tuning and threshold optimization.

- Best Model: Logistic Regression  
- Best CV Recall: ~0.81  
- Final Threshold: 0.3  
- Final Recall (Test): ~0.92  

This ensures that most churn customers are correctly identified.

In [15]:
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.3).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[557 478]
 [ 31 343]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.54      0.69      1035
           1       0.42      0.92      0.57       374

    accuracy                           0.64      1409
   macro avg       0.68      0.73      0.63      1409
weighted avg       0.81      0.64      0.66      1409



## Business Impact

This model helps telecom companies:

- Identify at-risk customers early  
- Take proactive retention actions  
- Reduce customer churn rate  
- Improve customer lifetime value (CLV)

In [19]:
import joblib

joblib.dump(best_model, 'churn_pred_model.pkl')

['churn_pred_model.pkl']

In [20]:
from google.colab import files
files.download('churn_pred_model.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>